In [11]:
import duckdb
import pandas as pd
from pathlib import Path

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')

In [12]:
class SetUp:

    def __init__(self):
        self._setup_db()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect()
        self.db.sql(f"ATTACH IF NOT EXISTS '{MY_DATABASE_FILE}' AS project")                     
        print(self.db.sql("SHOW ALL TABLES").df())
        return

In [13]:
class ETL(SetUp):

    def __init__(self):
        super().__init__()
        self.assemble_tables = {}
        return
    
    def sample_table(self):
        cols = ['Research_Profile', 'ACR', 'PUB', 'CIT', 'HCP', 'suma', 'coc', 'score', 'Group', 
                'author_id', 'orcid', 'fullname', 'works_count_endogenous', 'citations_endogenous', 'hca_endogenous', 
                'works_count_total', 'cited_by_count', 'hca_total', '2yr_mean_citedness', 'h_index', 'citations_total_oa']       
        sample = self.db.sql("SELECT * FROM project.sample_matched").df()[cols]
        sample = sample.sort_values(['Group', 'Research_Profile'], ascending=[False, True]).reset_index(drop=True)
        print(f'{sample.shape = }\n{sample.head()}')
        self.assemble_tables |= {'Domingo_matched': sample}
        return
    
    def works_table(self):
        # cols = [c.strip() for c in temp.split(" ") if c]
        cols = ['first_page', 'work_id', 'doi', 'title', 'publication_year', 'type', 
                'countries_distinct_count', 'institutions_distinct_count', 
                'fwci', 'cited_by_count', 'referenced_works_count', 
                'source_id', 'source_name', 'host_id', 'host_name']
        works = self.db.sql("SELECT * FROM project.works WHERE referenced_works_count > 0 and first_page != 'i'").df()[cols]
        works = works[works.referenced_works_count > 0].sort_values(['publication_year', 'fwci'], ascending=[True, False]).\
            drop(columns=['first_page']).reset_index(drop=True)
        print(f'{works.shape = }\n{works.head()}')
        self.assemble_tables |= {'works': works}
        return
    
    def authorships_table(self):
        sql = """SELECT a.work_id, author_id, author_name, orcid, institution_id, institution_name, country_code, a.type 
                    FROM project.works 
                    INNER JOIN project.authorships a
                    USING (work_id) 
                    WHERE referenced_works_count > 0 AND first_page != 'i'
                """
        authorships = self.db.sql(sql).df()
        print(f"{authorships.shape = }\n{authorships.head()}")
        self.assemble_tables |= {'authorships': authorships}
        return
    
    def references_table(self):
        sql = """
                SELECT c.work_id AS citer_id,
                        len(referenced_works) AS references_count,
                        referenced_works AS cited_id 
                    FROM project.cited c
                    INNER JOIN project.works w
                    USING (work_id)
                    WHERE referenced_works_count > 0 and first_page != 'i'
            """
        references = self.db.sql(sql).df()
        print(f'{references.shape = }\n{references.head()}')
        self.assemble_tables |= {'references': references}
        return
    
    def citations_table(self):
        sql = """
            SELECT cited_id,
                    count(citer_id) AS citations_count,
                    list(citer_id) AS citing_works 
                FROM
                    (SELECT work_id AS citer_id,
                            unnest(referenced_works) AS cited_id,
                        FROM project.cited) c
                LEFT JOIN project.works w
                ON c.cited_id = w.work_id
                WHERE referenced_works_count > 0 and first_page != 'i'
                GROUP BY cited_id
                ORDER BY citations_count DESC
            """
        citations = self.db.sql(sql).df()
        print(f'{citations.shape = }\n{citations.head()}')
        self.assemble_tables |= {'citations': citations}
        return

    def load_tables(self):
        for k, v in self.assemble_tables.items():
            with open(f'../DATA/tables_for_Domingo_{k}.csv', 'w') as writer:
                print(f'{k = } {v.shape = }\n{v.head()}')
                v.to_csv(writer, index=False)


In [14]:
def main():

    etl = ETL()
    etl.sample_table()
    etl.works_table()
    etl.authorships_table()
    etl.references_table()
    etl.citations_table()
    etl.load_tables()

    return

In [15]:
if __name__ == "__main__":
    main()
    print("DONE")

   database schema                             name  \
0   project   main              author_works_counts   
1   project   main                          authors   
2   project   main                      authorships   
3   project   main                 citation_summary   
4   project   main                            cited   
5   project   main                   domingo_sample   
6   project   main                   edge_list_both   
7   project   main           edge_list_institutions   
8   project   main                edge_list_sources   
9   project   main             hcp_count_endogenous   
10  project   main            institution_citations   
11  project   main                    pagerank_both   
12  project   main            pagerank_institutions   
13  project   main                 pagerank_sources   
14  project   main                   sample_matched   
15  project   main                     sample_names   
16  project   main                          samples   
17  projec